# Sistema predictivo de retención de clientes

En este notebook preparé la primera parte del proyecto. Primero cargué y revisé el CSV. Después limpié los datos, comparé tres modelos y guardé las probabilidades para analizarlas en la parte económica.

## 1. Preparé las librerías

Instalé las librerías que usé para el balanceo, los modelos y la conexión opcional con MySQL.

In [ ]:
%pip -q install imbalanced-learn xgboost sqlalchemy pymysql

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbalancedPipeline
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

SEMILLA = 42

## 2. Cargué y revisé el dataset

Usé el mismo CSV que aparece en el script de descarga de la profesora. También guardé una copia en el entorno de Colab.

In [ ]:
URL_DATASET = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Leí el CSV desde el enlace.
df_original = pd.read_csv(URL_DATASET)

# Guardé una copia en Colab para poder descargarla si la necesito.
df_original.to_csv("Telco_Customer_Churn.csv", index=False)

print("Filas y columnas:", df_original.shape)
display(df_original.head())
print("Columnas:")
print(df_original.columns.tolist())

In [ ]:
# Revisé que estuvieran las columnas principales antes de continuar.
columnas_necesarias = ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
faltantes = [columna for columna in columnas_necesarias if columna not in df_original.columns]

if faltantes:
    raise ValueError(f"Faltan columnas que necesito: {faltantes}")

print("Valores vacíos por columna:")
display(df_original.isna().sum().sort_values(ascending=False).head(10))

## 3. Limpié los datos

TotalCharges llegó como texto y algunas filas tenían el valor vacío. Convertí esa columna a número. En los clientes nuevos, con cero meses de antigüedad, tomé ese vacío como cero. Las otras columnas las completé dentro del modelo, para que el cálculo se hiciera usando solo los datos de entrenamiento.

In [ ]:
df = df_original.copy()

# Convertí TotalCharges a número. Los espacios vacíos quedaron como valores faltantes.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Si el cliente tenía cero meses y no tenía cargo total, tomé el valor como cero.
cliente_nuevo = (df["tenure"] == 0) & (df["TotalCharges"].isna())
df.loc[cliente_nuevo, "TotalCharges"] = 0

# Convertí la respuesta Yes/No a 1/0 para que los modelos la pudieran usar.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Cantidad de clientes por resultado:")
display(df["Churn"].value_counts().rename(index={0: "No se fue", 1: "Se fue"}))

In [ ]:
# Hice este gráfico para ver cuántas bajas había en los datos.
df["Churn"].value_counts().sort_index().plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["No se fue", "Se fue"], rotation=0)
plt.ylabel("Cantidad de clientes")
plt.title("Clientes por estado de churn")
plt.show()

## 4. Separé entrenamiento, validación y prueba

Usé tres grupos para que la prueba final quedara aparte. Entrené con el primer grupo, elegí el modelo con el segundo y medí el resultado final con el tercero. Así no usé la prueba final para elegir el modelo.

In [ ]:
# Dejé customerID fuera porque solo identifica al cliente.
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].astype(int)

# Separé primero el 20 por ciento para la prueba final.
X_entrenamiento_validacion, X_prueba, y_entrenamiento_validacion, y_prueba = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=y,
)

# Del 80 por ciento restante separé un 25 por ciento para validación.
X_entrenamiento, X_validacion, y_entrenamiento, y_validacion = train_test_split(
    X_entrenamiento_validacion,
    y_entrenamiento_validacion,
    test_size=0.25,
    random_state=SEMILLA,
    stratify=y_entrenamiento_validacion,
)

print("Entrenamiento:", len(X_entrenamiento))
print("Validación:", len(X_validacion))
print("Prueba:", len(X_prueba))

In [ ]:
# Separé las columnas numéricas de las columnas de texto.
columnas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
columnas_texto = X.select_dtypes(exclude=["number"]).columns.tolist()

# En números completé los faltantes con la mediana y escalé los valores.
preparar_numeros = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

# En texto usé el valor más frecuente y después convertí categorías a números.
preparar_texto = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore")),
])

# Junté los dos pasos para aplicarlos a sus columnas correspondientes.
preparar_datos = ColumnTransformer(transformers=[
    ("numeros", preparar_numeros, columnas_numericas),
    ("texto", preparar_texto, columnas_texto),
])

## 5. Comparé tres modelos

Elegí Regresión Logística como una primera referencia sencilla, Random Forest para comparar varios árboles y XGBoost para comparar un modelo de árboles que se construyen en secuencia. No hice una búsqueda grande de configuraciones; mantuve pocos parámetros para que el notebook fuera más fácil de revisar.

Puse el balanceo dentro del pipeline. Así se aplicó solo a los datos de entrenamiento y no a los de validación ni prueba.

In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEMILLA,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=SEMILLA,
        n_jobs=-1,
    ),
}

modelos_entrenados = {}
resultados = []

for nombre, modelo in modelos.items():
    # Primero preparé y balanceé el entrenamiento, y después entrené el modelo.
    pipeline = ImbalancedPipeline(steps=[
        ("preparar", preparar_datos),
        ("balancear", RandomOverSampler(random_state=SEMILLA)),
        ("modelo", modelo),
    ])

    pipeline.fit(X_entrenamiento, y_entrenamiento)
    probabilidad = pipeline.predict_proba(X_validacion)[:, 1]
    prediccion = (probabilidad >= 0.5).astype(int)

    modelos_entrenados[nombre] = pipeline
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_validacion, prediccion),
        "precision": precision_score(y_validacion, prediccion, zero_division=0),
        "recall": recall_score(y_validacion, prediccion, zero_division=0),
        "roc_auc": roc_auc_score(y_validacion, probabilidad),
    })

tabla_resultados = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)
display(tabla_resultados.round(3))

## 6. Elegí un modelo y revisé la prueba final

Tomé el modelo con el ROC AUC más alto en validación. Después lo medí una sola vez en el conjunto de prueba. El corte de 0,5 se usó solamente para estas métricas de clasificación; en la parte económica voy a buscar el umbral que dé mayor valor esperado.

In [ ]:
mejor_nombre = tabla_resultados.iloc[0]["modelo"]
mejor_modelo = modelos_entrenados[mejor_nombre]

probabilidad_validacion = mejor_modelo.predict_proba(X_validacion)[:, 1]
probabilidad_prueba = mejor_modelo.predict_proba(X_prueba)[:, 1]

prediccion_prueba = (probabilidad_prueba >= 0.5).astype(int)

print("Modelo elegido en validación:", mejor_nombre)
print("Resultados en prueba con corte 0,5")
print("Accuracy:", round(accuracy_score(y_prueba, prediccion_prueba), 3))
print("Precision:", round(precision_score(y_prueba, prediccion_prueba, zero_division=0), 3))
print("Recall:", round(recall_score(y_prueba, prediccion_prueba, zero_division=0), 3))
print("ROC AUC:", round(roc_auc_score(y_prueba, probabilidad_prueba), 3))

In [ ]:
# Guardé las probabilidades de validación y prueba para usarlas en la parte económica.
predicciones_validacion = pd.DataFrame({
    "conjunto": "validacion",
    "customer_id": df.loc[X_validacion.index, "customerID"].to_numpy(),
    "churn_real": y_validacion.to_numpy(),
    "cargo_mensual": df.loc[X_validacion.index, "MonthlyCharges"].to_numpy(),
    "probabilidad_churn": probabilidad_validacion,
})

predicciones_prueba = pd.DataFrame({
    "conjunto": "prueba",
    "customer_id": df.loc[X_prueba.index, "customerID"].to_numpy(),
    "churn_real": y_prueba.to_numpy(),
    "cargo_mensual": df.loc[X_prueba.index, "MonthlyCharges"].to_numpy(),
    "probabilidad_churn": probabilidad_prueba,
})

predicciones = pd.concat([predicciones_validacion, predicciones_prueba], ignore_index=True)
predicciones.to_csv("predicciones_para_revision.csv", index=False)

print("Guardé predicciones_para_revision.csv")
display(predicciones.head())

## 7. Cargué los datos limpios a MySQL con SQLAlchemy

Dejé la carga apagada para que el notebook se pueda ejecutar en Colab sin las credenciales de una base. Para cargar, primero ejecuté el script SQL en MySQL Workbench y después configuré los datos de conexión. El servidor tiene que ser accesible desde el entorno donde ejecuto Python.

In [ ]:
CARGAR_EN_MYSQL = False

if CARGAR_EN_MYSQL:
    from getpass import getpass
    from sqlalchemy import create_engine
    from sqlalchemy.engine import URL

    # Completé estos datos con la conexión del servidor MySQL.
    MYSQL_HOST = os.environ.get("MYSQL_HOST", input("Host del servidor MySQL: "))
    MYSQL_USER = os.environ.get("MYSQL_USER", input("Usuario MySQL: "))
    MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD") or getpass("Contraseña MySQL: ")

    conexion_mysql = URL.create(
        "mysql+pymysql",
        username=MYSQL_USER,
        password=MYSQL_PASSWORD,
        host=MYSQL_HOST,
        port=3306,
        database="churn_pp2",
    )
    engine = create_engine(conexion_mysql)

    # Cambié los nombres para que coincidieran con las columnas del SQL.
    datos_sql = df.rename(columns={
        "customerID": "customer_id",
        "SeniorCitizen": "senior_citizen",
        "PhoneService": "phone_service",
        "MultipleLines": "multiple_lines",
        "InternetService": "internet_service",
        "OnlineSecurity": "online_security",
        "OnlineBackup": "online_backup",
        "DeviceProtection": "device_protection",
        "TechSupport": "tech_support",
        "StreamingTV": "streaming_tv",
        "StreamingMovies": "streaming_movies",
        "Contract": "contract",
        "PaperlessBilling": "paperless_billing",
        "PaymentMethod": "payment_method",
        "MonthlyCharges": "monthly_charges",
        "TotalCharges": "total_charges",
        "Churn": "churn",
    })

    datos_cliente = datos_sql[[
        "customer_id", "gender", "senior_citizen", "partner", "dependents"
    ]]
    datos_servicio = datos_sql[[
        "customer_id", "tenure", "phone_service", "multiple_lines",
        "internet_service", "online_security", "online_backup",
        "device_protection", "tech_support", "streaming_tv",
        "streaming_movies", "contract", "paperless_billing",
        "payment_method", "monthly_charges", "total_charges", "churn"
    ]]

    # Cargué primero los clientes para respetar la clave foránea.
    with engine.begin() as conexion:
        datos_cliente.to_sql("dim_customer", conexion, if_exists="append", index=False, chunksize=1000)
        datos_servicio.to_sql("fact_subscription", conexion, if_exists="append", index=False, chunksize=1000)

    print("Terminé la carga en MySQL.")
else:
    print("Dejé la carga de MySQL apagada. El resto del notebook sí se ejecutó.")